# Graph Pattern Example

This notebook demonstrates the Graph pattern from Chapter 4.

## Overview
We'll build an e-commerce order processing pipeline where orders follow explicit, auditable paths:
- **Risk Analyzer**: Categorizes orders by risk level
- **Manual Review Agent**: Reviews high-value orders
- **Fraud Detection Agent**: Checks suspicious orders
- **Inventory Agent**: Checks stock availability
- **Shipping Agent**: Processes shipments
- **Refund Agent**: Handles out-of-stock refunds

Unlike Swarm, the routing is explicit in code — every path is visible and auditable.

## Setup

In [ ]:
# Uncomment to install
# !pip install strands-agents

In [ ]:
from strands import Agent
from strands.multiagent import GraphBuilder

## Create Specialized Agents

Each agent handles one step in the order processing pipeline.

In [ ]:
risk_analyzer = Agent(
    name="risk_analyzer",
    system_prompt="""Analyze order risk level. Categorize each order as:
    - HIGH_VALUE: Orders over $1,000
    - SUSPICIOUS: Mismatched billing/shipping address, new account with large order, or known fraud patterns
    - LOW_RISK: Regular orders from trusted customers
    Always include the risk category in your response."""
)

manual_review_agent = Agent(
    name="manual_review_agent",
    system_prompt="Review high-value orders. Verify customer identity, check order history, and approve or flag for further review."
)

fraud_detection_agent = Agent(
    name="fraud_detection_agent",
    system_prompt="Run advanced fraud detection on suspicious orders. Check for known fraud patterns, velocity checks, and device fingerprinting."
)

inventory_agent = Agent(
    name="inventory_agent",
    system_prompt="""Check inventory availability for the order items. Respond with:
    - IN_STOCK: All items available
    - OUT_OF_STOCK: One or more items unavailable
    Always include the stock status in your response."""
)

shipping_agent = Agent(
    name="shipping_agent",
    system_prompt="Generate shipping labels and process the shipment. Confirm tracking number and estimated delivery date."
)

refund_agent = Agent(
    name="refund_agent",
    system_prompt="Process refund for out-of-stock items. Notify the customer and initiate payment reversal."
)

print("✓ All agents created")

## Define Routing Conditions

These functions check the output from previous nodes to decide which path to follow. The logic is explicit in code — not hidden in LLM reasoning.

In [ ]:
def is_high_value(state):
    """Route to manual review if order is high value."""
    result = str(state.results.get("risk_analysis", "")).upper()
    return "HIGH_VALUE" in result

def is_suspicious(state):
    """Route to fraud detection if order is suspicious."""
    result = str(state.results.get("risk_analysis", "")).upper()
    return "SUSPICIOUS" in result

def is_low_risk(state):
    """Route directly to inventory if order is low risk."""
    result = str(state.results.get("risk_analysis", "")).upper()
    return "LOW_RISK" in result

def is_in_stock(state):
    """Route to shipping if items are in stock."""
    result = str(state.results.get("inventory_check", "")).upper()
    return "IN_STOCK" in result

def is_out_of_stock(state):
    """Route to refund if items are out of stock."""
    result = str(state.results.get("inventory_check", "")).upper()
    return "OUT_OF_STOCK" in result

print("✓ Routing conditions defined")

## Build the Graph

Add agents as nodes, then connect them with edges. Conditional edges route orders based on risk level and stock availability.

```
                    ┌─ HIGH_VALUE ──→ Manual Review ─┐
Risk Analyzer ──────┼─ SUSPICIOUS ──→ Fraud Check ───┼──→ Inventory ──┬─ IN_STOCK ────→ Shipping
                    └─ LOW_RISK ─────────────────────┘               └─ OUT_OF_STOCK ─→ Refund
```

In [ ]:
builder = GraphBuilder()

# Add nodes
builder.add_node(risk_analyzer, "risk_analysis")
builder.add_node(manual_review_agent, "manual_review")
builder.add_node(fraud_detection_agent, "fraud_check")
builder.add_node(inventory_agent, "inventory_check")
builder.add_node(shipping_agent, "shipping")
builder.add_node(refund_agent, "refund")

# Add conditional edges from risk analysis
builder.add_edge("risk_analysis", "manual_review", condition=is_high_value)
builder.add_edge("risk_analysis", "fraud_check", condition=is_suspicious)
builder.add_edge("risk_analysis", "inventory_check", condition=is_low_risk)

# After review/fraud check, proceed to inventory
builder.add_edge("manual_review", "inventory_check")
builder.add_edge("fraud_check", "inventory_check")

# After inventory, ship or refund
builder.add_edge("inventory_check", "shipping", condition=is_in_stock)
builder.add_edge("inventory_check", "refund", condition=is_out_of_stock)

# Set entry point and build
builder.set_entry_point("risk_analysis")
graph = builder.build()

print("✓ Graph built")

## Process Orders

Each order follows a different path through the graph based on its risk level.

In [ ]:
# High-value order → manual review → inventory → shipping
print("Order 1: High-value order")
print("="*80)
result = graph("Order #12345: $1,500 laptop from new customer, billing and shipping address match")
print(result)

In [ ]:
# Low-risk order → straight to inventory → shipping
print("Order 2: Low-risk order")
print("="*80)
result = graph("Order #67890: $45 book from returning customer with 50+ previous orders")
print(result)

In [ ]:
# Suspicious order → fraud check → inventory → shipping/refund
print("Order 3: Suspicious order")
print("="*80)
result = graph("Order #11111: $800 electronics, new account created 5 minutes ago, shipping to different country than billing")
print(result)

## Key Takeaways

1. **Explicit routing**: Every path is visible in code — not hidden in LLM reasoning
2. **Auditable**: If someone asks "how do you handle fraud?", point to the graph structure
3. **Directed Acyclic Graph**: Arrows only point forward, no loops — every order reaches an end state
4. **Conditional edges**: Route based on data, not LLM interpretation
5. **Use when**: You have established procedures (compliance, approval chains, order processing)
6. **Don't use when**: The path can't be predicted upfront — use Swarm instead